# Building a multimodal workflow with a local Gemma model

This notebook prepares images, analyzes them with Gemma 4 through Ollama, and synthesizes the resulting records into a trip memory. Keep the input photos outside this repository.

In [ ]:
# If needed, run this once in a notebook cell:
# %pip install -r ../requirements.txt

from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from multimodal_workflow import (
    analyze_photo,
    prepare_photo,
    synthesize_trip,
)

In [ ]:
# Change this to a private directory on your machine.
PHOTO_DIR = Path(r'C:\path\to\your\image_collection')
MODEL = 'gemma4:e4b-128k'
MAX_EDGE_PX = 1280

photo_paths = sorted(
    path for path in PHOTO_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
)
len(photo_paths), photo_paths[:2]

In [ ]:
# Stage 1: deterministic image preparation and metadata extraction.
prepared = prepare_photo(photo_paths[0], max_edge_px=MAX_EDGE_PX)
prepared.metadata

In [ ]:
# Stage 2: multimodal structured photo analysis.
photo_analysis = analyze_photo(prepared, model=MODEL)
photo_analysis.model_dump()

In [ ]:
# Run Stage 2 for every image, retaining the filename as evidence ID.
photo_memories = []
for image_path in photo_paths:
    item = prepare_photo(image_path, max_edge_px=MAX_EDGE_PX)
    analysis = analyze_photo(item, model=MODEL)
    photo_memories.append({
        'photo_id': item.photo_id,
        'metadata': item.metadata,
        'analysis': analysis.model_dump(),
    })

len(photo_memories)

In [ ]:
# Stage 3: text-only trip-level synthesis.
trip_memory = synthesize_trip(photo_memories, model=MODEL)
trip_memory.model_dump()

In [ ]:
# Save a portable result outside Git's tracked files.
output_path = PROJECT_ROOT / 'outputs' / 'trip_memory.json'
output_path.parent.mkdir(exist_ok=True)
output_path.write_text(
    json.dumps({
        'photo_memories': photo_memories,
        'trip_memory': trip_memory.model_dump(),
    }, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
output_path